# Activation Patching

# Activation Patching

## What This Is
Activation patching (causal tracing) is the core experimental tool of mech interp. It answers: *which model components causally mediate a specific output?*

**Protocol**:
1. Run model on a *clean* input, cache all activations
2. Run model on a *corrupt* input (e.g., change a fact)
3. For each component, patch its activation from the clean run into the corrupt run
4. Measure how much the output changes — components with large effect are causally important

**ROME experiments**: Meng et al. (2022) used activation patching on GPT models to identify that factual recall is mediated by specific MLP layers at the subject's last token position. This led to the ROME editing technique.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Causal tracing for factual recall localization
# Simplified version of the ROME experiments

D = 16
N_LAYERS = 4
VOCAB = 40
SEQ = 6

class CausalTracingModel:
    def __init__(self):
        np.random.seed(0)
        self.W_emb = np.random.randn(VOCAB, D) * 0.2
        self.layers = [
            {'W_attn': np.random.randn(D, D) * 0.15,
             'W_ff': np.random.randn(D, 2*D) * 0.15,
             'W_ff_out': np.random.randn(2*D, D) * 0.15}
            for _ in range(N_LAYERS)
        ]
        # Encode a 'fact': token 5 always predicts token 20 at layer 2
        # Encode this by biasing W_ff at layer 2
        fact_key = self.W_emb[5]  # subject token embedding
        fact_val = self.W_emb[20]  # object token embedding
        self.layers[2]['W_ff_bias'] = np.outer(fact_key @ self.layers[2]['W_ff'][:, :5], fact_val[:5])
        self.W_unembed = self.W_emb.T * 0.3
    
    def forward(self, tokens, corrupt_layer=None, corrupt_pos=None,
                clean_activations=None):
        x = self.W_emb[tokens]
        cache = [x.copy()]
        
        for l in range(N_LAYERS):
            # Attention (simplified single-head)
            attn_out = x @ self.layers[l]['W_attn']
            x = x + 0.3 * attn_out
            # MLP
            mlp_out = np.maximum(0, x @ self.layers[l]['W_ff']) @ self.layers[l]['W_ff_out']
            x = x + 0.3 * mlp_out
            
            # Apply corruption: replace layer l activation at pos with clean version
            if (corrupt_layer == l and corrupt_pos is not None and
                    clean_activations is not None):
                x[corrupt_pos] = clean_activations[l+1][corrupt_pos]
            
            cache.append(x.copy())
        
        return x @ self.W_unembed, cache

model = CausalTracingModel()

# Clean run: subject at position 2
tokens_clean = [1, 3, 5, 7, 9, 11]  # token 5 is the subject
tokens_corrupt = [1, 3, 6, 7, 9, 11]  # token 6 instead of 5

logits_clean, cache_clean = model.forward(tokens_clean)
logits_corrupt, _ = model.forward(tokens_corrupt)

target_token = 20  # expected prediction
clean_prob = float(np.exp(logits_clean[-1, target_token]) / np.exp(logits_clean[-1]).sum())
corrupt_prob = float(np.exp(logits_corrupt[-1, target_token]) / np.exp(logits_corrupt[-1]).sum())

print(f'Fact: token 5 -> token 20')
print(f'Clean P(token 20): {clean_prob:.4f}')
print(f'Corrupt P(token 20): {corrupt_prob:.4f}')
print(f'Causal gap: {clean_prob - corrupt_prob:.4f}')

print('\nActivation Patching: restore clean activation at each layer')
print(f'{'Layer':<8} {'Restored P(20)':>15} {'Recovery':>10}')
print('-' * 38)
for layer in range(N_LAYERS):
    logits_patched, _ = model.forward(tokens_corrupt, corrupt_layer=layer,
                                       corrupt_pos=2, clean_activations=cache_clean)
    patched_prob = float(np.exp(logits_patched[-1, target_token]) / np.exp(logits_patched[-1]).sum())
    recovery = (patched_prob - corrupt_prob) / (clean_prob - corrupt_prob + 1e-8)
    print(f'  L{layer}      {patched_prob:>15.4f} {recovery:>10.3f}')
